In [16]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json

load_dotenv(override=True)

True

In [17]:
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)

In [18]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

VALID_ORDERS = ["OR-1001", "OR-1024", "OR-5500", "OR-9999"]

complaint_list = []

In [19]:
def fetch_menu():
    return menu

def validate_order_id(order_id):
    """Validates if the provided order_id exists in the system."""
    # Check format or existence against valid orders database
    is_valid = order_id in VALID_ORDERS
    if is_valid:
        return {"valid": True, "message": f"Order ID {order_id} is valid."}
    else:
        return {
            "valid": False,
            "message": f"Order ID {order_id} is invalid. Please ask the user to provide a correct order ID.",
        }
        
def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate if an order ID exists in the system before logging a complaint",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID provided by the customer (e.g., OR-1024)",
                    }
                },
                "required": ["order_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [20]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, or recommendations.
        2. **`validate_order_id`** — MANDATORY before logging a complaint. Call this tool first to check the provided order_id.
           - If the validation fails (returns valid: false), inform the user: **"Please provide a correct order ID."** Do NOT log the complaint.
        3. **`log_complaint`** — Call ONLY after `validate_order_id` returns valid: true.

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]

In [23]:
from IPython.display import Markdown

In [24]:
function_names = {
    "fetch_menu": fetch_menu,
    "validate_order_id": validate_order_id,
    "log_complaint": log_complaint
}


def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    completion = llm_client.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools
    )
   
    reply = completion.choices[0].message

    while reply.tool_calls:
        messages.append(reply.model_dump(exclude_none=True))

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)
            print("Tool executed:", call.function.name, args, "Result:", result)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(result),
                }
            )

        completion = llm_client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
        reply = completion.choices[0].message

    messages.append({"role": "assistant", "content": reply.content})
    return Markdown(reply.content)

In [26]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-5500, phone is 0322342423")

Tool executed: validate_order_id {'order_id': 'OR-5500'} Result: {'valid': True, 'message': 'Order ID OR-5500 is valid.'}
Tool executed: log_complaint {'order_id': 'OR-5500', 'customer_name': 'Sam', 'phone': '0322342423', 'complaint': 'Roll served instead of burger, wrong bill'} Result: {'message': 'Saved! Total Complaint: 1'}


Thank you, Sam! Your complaint has been logged successfully. We’ll look into the roll‑instead‑of‑burger issue and the incorrect bill for order OR‑5500. Appreciate your patience!